In [20]:
import pandas as pd
import numpy as np

In [21]:
data_1 =pd.read_excel( "DATASETS\PGCB_date_power_demand.xlsx")


In [4]:
data_2=pd.read_csv("powergrid_data.csv",header=1)


In [5]:
print(data_1.columns)
print(data_2.columns)

Index(['datetime', 'generation_mw', 'demand_mw', 'load_shedding', 'gas',
       'liquid_fuel', 'coal', 'hydro', 'solar', 'wind', 'india_bheramara_hvdc',
       'india_tripura', 'india_adani', 'nepal', 'remarks'],
      dtype='object')
Index(['Date', 'Time', 'Generation(MW)', 'Demand(MW)', 'Loadshed', 'Gas',
       'Liquid Fuel', 'Coal', 'Hydro', 'Solar', 'Wind', 'Bheramara HVDC',
       'Tripura', 'Adani', 'Nepal', 'Remarks'],
      dtype='object')


In [6]:
weather_data=pd.read_csv("DATASETS\DHAKA HOURLY weather.csv")
weather_data.columns

Index(['YEAR', 'MO', 'DY', 'HR', 'T2M', 'RH2M', 'WS10M'], dtype='object')

In [9]:
rain_data = pd.read_csv(r"DATASETS\bgd-rainfall-subnat-full.csv")
rain_data.columns

Index(['date', 'adm_level', 'adm_id', 'PCODE', 'n_pixels', 'rfh', 'rfh_avg',
       'r1h', 'r1h_avg', 'r3h', 'r3h_avg', 'rfq', 'r1q', 'r3q', 'version'],
      dtype='object')

In [15]:
# Normalize and align power datasets
data_1 = data_1.copy()
data_2 = data_2.copy()

data_1["datetime"] = pd.to_datetime(data_1["datetime"], errors="coerce")
data_1 = data_1[
    data_1["datetime"].dt.minute.eq(0)
    & data_1["datetime"].dt.second.eq(0)
].copy()
data_1["Date"] = data_1["datetime"].dt.date.astype("string")
data_1["Time"] = data_1["datetime"].dt.time.astype("string")

map_1_to_2 = {
    "generation_mw": "Generation(MW)",
    "demand_mw": "Demand(MW)",
    "load_shedding": "Loadshed",
    "gas": "Gas",
    "liquid_fuel": "Liquid Fuel",
    "coal": "Coal",
    "hydro": "Hydro",
    "solar": "Solar",
    "wind": "Wind",
    "india_bheramara_hvdc": "Bheramara HVDC",
    "india_tripura": "Tripura",
    "india_adani": "Adani",
    "nepal": "Nepal",
    "remarks": "Remarks",
}

data_1_aligned = data_1.rename(columns=map_1_to_2)

data_2["Date"] = pd.to_datetime(data_2["Date"], errors="coerce").dt.date.astype("string")
data_2["Time"] = pd.to_datetime(data_2["Time"].astype(str), errors="coerce").dt.time.astype("string")
data_2["datetime"] = pd.to_datetime(
    data_2["Date"].astype(str) + " " + data_2["Time"].astype(str),
    errors="coerce",
)
data_2 = data_2[
    data_2["datetime"].dt.minute.eq(0)
    & data_2["datetime"].dt.second.eq(0)
].copy()

power_cols = list(data_2.columns)
data_1_aligned = data_1_aligned.reindex(columns=power_cols)

data_1_aligned["datetime"] = pd.to_datetime(
    data_1_aligned["Date"].astype(str) + " " + data_1_aligned["Time"].astype(str),
    errors="coerce",
)

C:\Users\defaultuser0.LAPTOP-LRB3T941\AppData\Local\Temp\ipykernel_7968\98128009.py:33: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  data_2["Time"] = pd.to_datetime(data_2["Time"].astype(str), errors="coerce").dt.time.astype("string")


In [16]:
# Combine and dedupe by datetime
power_combined = pd.concat([data_1_aligned, data_2], ignore_index=True)
power_combined = power_combined.drop_duplicates(subset=["datetime"], keep="first")
power_combined = power_combined.sort_values("datetime").reset_index(drop=True)

power_combined[["datetime", "Date", "Time"]].head()

,datetime,Date,Time
0,2015-04-19 00:00:00,2015-04-19,00:00:00
1,2015-04-19 01:00:00,2015-04-19,01:00:00
2,2015-04-19 02:00:00,2015-04-19,02:00:00
3,2015-04-19 03:00:00,2015-04-19,03:00:00
4,2015-04-19 04:00:00,2015-04-19,04:00:00


In [17]:
# Prepare weather data and merge on datetime
weather = weather_data.copy()
weather["datetime"] = pd.to_datetime(
    weather["YEAR"].astype(str)
    + "-"
    + weather["MO"].astype(int).astype(str).str.zfill(2)
    + "-"
    + weather["DY"].astype(int).astype(str).str.zfill(2)
    + " "
    + weather["HR"].astype(int).astype(str).str.zfill(2)
    + ":00:00",
    errors="coerce",
)
weather = weather[["datetime", "T2M", "RH2M", "WS10M"]]

power_weather = power_combined.merge(weather, on="datetime", how="left")

power_weather[["datetime", "T2M", "RH2M", "WS10M"]].head()

,datetime,T2M,RH2M,WS10M
0,2015-04-19 00:00:00,25.39,91.45,4.62
1,2015-04-19 01:00:00,25.06,94.06,4.59
2,2015-04-19 02:00:00,24.81,96.04,4.28
3,2015-04-19 03:00:00,24.56,97.52,3.59
4,2015-04-19 04:00:00,24.43,97.86,3.54


In [60]:
weather_2 = pd.read_csv("DATASETS\\wined 2.csv",header=11)
weather_3 = pd.read_csv("DATASETS\\wind prec.csv",header=11)
print(weather_2.columns)
print(weather_3.columns)

Index(['YEAR', 'MO', 'DY', 'HR', 'WD10M', 'WD2M', 'WS2M'], dtype='object')
Index(['YEAR', 'MO', 'DY', 'HR', 'QV2M', 'PRECTOTCORR', 'PS'], dtype='object')


In [47]:
rain_data.isna().sum()

date           0
adm_level      0
adm_id         0
PCODE          0
n_pixels       0
rfh            0
rfh_avg        0
r1h          148
r1h_avg      148
r3h          592
r3h_avg      592
rfq            0
r1q          148
r3q          592
version        0
dtype: int64

In [62]:
# Merge additional weather data (wind and precipitation)
wind_prec = pd.read_csv(r"DATASETS\wind prec.csv", header=11)
wind_2 = pd.read_csv(r"DATASETS\wined 2.csv", header=11)

def process_nasa_df(df):
    df["datetime"] = pd.to_datetime(
        df["YEAR"].astype(str)
        + "-"
        + df["MO"].astype(int).astype(str).str.zfill(2)
        + "-"
        + df["DY"].astype(int).astype(str).str.zfill(2)
        + " "
        + df["HR"].astype(int).astype(str).str.zfill(2)
        + ":00:00",
        errors="coerce",
    )
    return df

wind_prec = process_nasa_df(wind_prec)
wind_2 = process_nasa_df(wind_2)

# Merge the two new dataframes
extra_weather = wind_prec.merge(
    wind_2[["datetime", "WD10M", "WD2M", "WS2M"]], 
    on="datetime", 
    how="left"
)

# Merge into power_weather
power_weather = power_weather.merge(
    extra_weather[["datetime", "QV2M", "PRECTOTCORR", "PS", "WD10M", "WD2M", "WS2M"]],
    on="datetime",
    how="left"
)

power_weather.head()

,Date,Time,Generation(MW),Demand(MW),Loadshed,Gas,Liquid Fuel,Coal,Hydro,Solar,...,T2M,RH2M,WS10M,date_only,QV2M,PRECTOTCORR,PS,WD10M,WD2M,WS2M
0,2015-04-19,00:00:00,4821.0,4821,0,0,0,0,0,NaN,...,25.39,91.45,4.62,2015-04-19,18.56,13.50,100.56,175.4,175.5,3.22
1,2015-04-19,01:00:00,3612.0,3612,0,0,0,0,0,NaN,...,25.06,94.06,4.59,2015-04-19,18.73,20.24,100.50,173.6,173.5,3.20
2,2015-04-19,02:00:00,3727.0,3727,0,0,0,0,0,NaN,...,24.81,96.04,4.28,2015-04-19,18.85,28.97,100.47,172.6,172.7,2.97
3,2015-04-19,03:00:00,3632.0,3632,0,0,0,0,0,NaN,...,24.56,97.52,3.59,2015-04-19,18.85,37.40,100.50,165.7,165.6,2.46
4,2015-04-19,04:00:00,3641.0,3641,0,0,0,0,0,NaN,...,24.43,97.86,3.54,2015-04-19,18.76,38.72,100.54,152.2,152.3,2.41


In [63]:
# Prepare rain data (date-level) and merge (prefer version == 'final')
rain = rain_data.copy()
rain["date_only"] = pd.to_datetime(rain["date"], errors="coerce").dt.date
rain = rain.drop(columns=["date"])

# Prefer final when available; otherwise keep other versions for that date
rain["version_norm"] = rain["version"].astype(str).str.lower()
rain["version_rank"] = np.where(rain["version_norm"].eq("final"), 0, 1)
rain = rain.sort_values(["date_only", "version_rank"]).drop_duplicates(subset=["date_only"], keep="first")
rain = rain.drop(columns=["version_norm", "version_rank"])

# Expand 1/11/21 monthly snapshots to daily rows
rain_expanded = []
for _, row in rain.iterrows():
    base_date = pd.to_datetime(row["date_only"])
    if pd.isna(base_date):
        continue
    day = base_date.day
    if day not in (1, 11, 21):
        continue
    start = base_date
    if day == 1:
        end = base_date + pd.Timedelta(days=9)
    elif day == 11:
        end = base_date + pd.Timedelta(days=9)
    else:
        end = base_date + pd.offsets.MonthEnd(0)
    for d in pd.date_range(start=start, end=end, freq="D"):
        new_row = row.copy()
        new_row["date_only"] = d.date()
        rain_expanded.append(new_row)

rain_daily = pd.DataFrame(rain_expanded)
rain_daily = rain_daily.drop_duplicates(subset=["date_only"], keep="first")

power_weather["date_only"] = power_weather["datetime"].dt.date
final_data = power_weather.merge(rain_daily, on="date_only", how="left").drop(columns=["date_only"])

final_data.head()

,Date,Time,Generation(MW),Demand(MW),Loadshed,Gas,Liquid Fuel,Coal,Hydro,Solar,...,rfh,rfh_avg,r1h,r1h_avg,r3h,r3h_avg,rfq,r1q,r3q,version
0,2015-04-19,00:00:00,4821.0,4821,0,0,0,0,0,NaN,...,9.818182,29.618181,110.98182,97.63697,146.14545,139.18788,42.804626,113.00199,104.82534,final
1,2015-04-19,01:00:00,3612.0,3612,0,0,0,0,0,NaN,...,9.818182,29.618181,110.98182,97.63697,146.14545,139.18788,42.804626,113.00199,104.82534,final
2,2015-04-19,02:00:00,3727.0,3727,0,0,0,0,0,NaN,...,9.818182,29.618181,110.98182,97.63697,146.14545,139.18788,42.804626,113.00199,104.82534,final
3,2015-04-19,03:00:00,3632.0,3632,0,0,0,0,0,NaN,...,9.818182,29.618181,110.98182,97.63697,146.14545,139.18788,42.804626,113.00199,104.82534,final
4,2015-04-19,04:00:00,3641.0,3641,0,0,0,0,0,NaN,...,9.818182,29.618181,110.98182,97.63697,146.14545,139.18788,42.804626,113.00199,104.82534,final


In [64]:
final_data['day'] = final_data['datetime'].dt.day
final_data['month'] = final_data['datetime'].dt.month
final_data['year'] = final_data['datetime'].dt.year

# Move day, month, year to the front
cols = ['day', 'month', 'year'] + [c for c in final_data.columns if c not in ['day', 'month', 'year']]
final_data = final_data[cols]

final_data.head()

,day,month,year,Date,Time,Generation(MW),Demand(MW),Loadshed,Gas,Liquid Fuel,...,rfh,rfh_avg,r1h,r1h_avg,r3h,r3h_avg,rfq,r1q,r3q,version
0,19,4,2015,2015-04-19,00:00:00,4821.0,4821,0,0,0,...,9.818182,29.618181,110.98182,97.63697,146.14545,139.18788,42.804626,113.00199,104.82534,final
1,19,4,2015,2015-04-19,01:00:00,3612.0,3612,0,0,0,...,9.818182,29.618181,110.98182,97.63697,146.14545,139.18788,42.804626,113.00199,104.82534,final
2,19,4,2015,2015-04-19,02:00:00,3727.0,3727,0,0,0,...,9.818182,29.618181,110.98182,97.63697,146.14545,139.18788,42.804626,113.00199,104.82534,final
3,19,4,2015,2015-04-19,03:00:00,3632.0,3632,0,0,0,...,9.818182,29.618181,110.98182,97.63697,146.14545,139.18788,42.804626,113.00199,104.82534,final
4,19,4,2015,2015-04-19,04:00:00,3641.0,3641,0,0,0,...,9.818182,29.618181,110.98182,97.63697,146.14545,139.18788,42.804626,113.00199,104.82534,final


In [65]:
columns_to_fill = ["Solar" ,"Wind" ,"Adani" ,"Nepal" ]
cols_to_drop = ["Remarks"]
final_data[columns_to_fill] = final_data[columns_to_fill].fillna(0)
final_data = final_data.drop(columns=cols_to_drop)
final_data.head()

,day,month,year,Date,Time,Generation(MW),Demand(MW),Loadshed,Gas,Liquid Fuel,...,rfh,rfh_avg,r1h,r1h_avg,r3h,r3h_avg,rfq,r1q,r3q,version
0,19,4,2015,2015-04-19,00:00:00,4821.0,4821,0,0,0,...,9.818182,29.618181,110.98182,97.63697,146.14545,139.18788,42.804626,113.00199,104.82534,final
1,19,4,2015,2015-04-19,01:00:00,3612.0,3612,0,0,0,...,9.818182,29.618181,110.98182,97.63697,146.14545,139.18788,42.804626,113.00199,104.82534,final
2,19,4,2015,2015-04-19,02:00:00,3727.0,3727,0,0,0,...,9.818182,29.618181,110.98182,97.63697,146.14545,139.18788,42.804626,113.00199,104.82534,final
3,19,4,2015,2015-04-19,03:00:00,3632.0,3632,0,0,0,...,9.818182,29.618181,110.98182,97.63697,146.14545,139.18788,42.804626,113.00199,104.82534,final
4,19,4,2015,2015-04-19,04:00:00,3641.0,3641,0,0,0,...,9.818182,29.618181,110.98182,97.63697,146.14545,139.18788,42.804626,113.00199,104.82534,final


In [66]:
final_data.isna().sum()

day                 0
month               0
year                0
Date                0
Time                0
Generation(MW)      0
Demand(MW)          0
Loadshed            0
Gas                 0
Liquid Fuel         0
Coal                0
Hydro               0
Solar               0
Wind                0
Bheramara HVDC      0
Tripura             0
Adani               0
Nepal               0
datetime            0
T2M               431
RH2M              431
WS10M             431
QV2M              431
PRECTOTCORR       431
PS                431
WD10M             431
WD2M              431
WS2M              431
adm_level         431
adm_id            431
PCODE             431
n_pixels          431
rfh               431
rfh_avg           431
r1h               431
r1h_avg           431
r3h               431
r3h_avg           431
rfq               431
r1q               431
r3q               431
version           431
dtype: int64

In [67]:
bangladesh_economy_comprehensive = {
    "metadata": {
        "country": "Bangladesh",
        "units": {
            "nominal": "Billions of Current US Dollars (USD)",
            "ppp": "Billions of Current International Dollars (Int$)",
            "per_capita": "Current US Dollars (USD) / International Dollars (Int$)"
        },
        "sources": [
            "World Bank (WDI)", 
            "IMF World Economic Outlook (Oct 2024)", 
            "Bangladesh Bureau of Statistics (BBS)", 
            "CEIC Data"
        ],
        "note": "2024-2025 data are IMF/World Bank projections. GNI is used as the modern equivalent of GNP."
    },
    "years": [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025],
    
    "indicators": {
        # GDP Nominal: The standard measure of economic output at current exchange rates.
        "gdp_nominal_billions_usd": [
            195.15,  # 2015
            265.23,  # 2016
            293.73,  # 2017
            321.36,  # 2018
            351.23,  # 2019
            373.98,  # 2020
            416.27,  # 2021
            460.13,  # 2022
            437.42,  # 2023 (Correction due to exchange rate depreciation)
            451.47,  # 2024 (IMF Estimate)
            481.86   # 2025 (IMF Projection)
        ],

        # GDP PPP: Economic output adjusted for cost of living/purchasing power.
        # This is often seen as a better measure of standard of living potential.
        "gdp_ppp_billions_intl_dollar": [
            671.36,   # 2015
            736.38,   # 2016
            791.89,   # 2017
            897.77,   # 2018
            997.25,   # 2019
            1104.32,  # 2020
            1247.56,  # 2021
            1430.21,  # 2022
            1568.54,  # 2023
            1692.74,  # 2024 (IMF Estimate)
            1801.05   # 2025 (IMF Projection)
        ],

        # GNI (formerly GNP): GDP + Net income from abroad (remittances, etc.)
        "gni_billions_usd": [
            203.80,  # 2015
            230.10,  # 2016
            258.40,  # 2017
            286.70,  # 2018
            316.50,  # 2019
            340.10,  # 2020
            436.75,  # 2021
            483.37,  # 2022
            493.93,  # 2023
            478.50,  # 2024 (Estimate)
            495.20   # 2025 (Projection)
        ],

        # GDP Per Capita (Nominal): Average income per person in USD.
        "gdp_per_capita_nominal_usd": [
            1248,  # 2015
            1401,  # 2016
            1563,  # 2017
            1698,  # 2018
            1855,  # 2019
            1968,  # 2020
            2457,  # 2021
            2687,  # 2022
            2528,  # 2023
            2619,  # 2024 (IMF Estimate)
            2734   # 2025 (IMF Projection)
        ],

        # GDP Per Capita (PPP): Average income adjusted for local purchasing power.
        "gdp_per_capita_ppp_intl_dollar": [
            3980,  # 2015
            4620,  # 2016
            4960,  # 2017
            5564,  # 2018
            6116,  # 2019
            6705,  # 2020
            7486,  # 2021
            8493,  # 2022
            9219,  # 2023
            9747,  # 2024 (IMF Estimate)
            10258  # 2025 (IMF Projection)
        ]
    }
}

# Convert dictionary to DataFrame and merge onto final_data
econ_df = pd.DataFrame(bangladesh_economy_comprehensive["indicators"])
econ_df["year"] = bangladesh_economy_comprehensive["years"]

final_data = final_data.merge(econ_df, on="year", how="left")
final_data.head()

,day,month,year,Date,Time,Generation(MW),Demand(MW),Loadshed,Gas,Liquid Fuel,...,r3h_avg,rfq,r1q,r3q,version,gdp_nominal_billions_usd,gdp_ppp_billions_intl_dollar,gni_billions_usd,gdp_per_capita_nominal_usd,gdp_per_capita_ppp_intl_dollar
0,19,4,2015,2015-04-19,00:00:00,4821.0,4821,0,0,0,...,139.18788,42.804626,113.00199,104.82534,final,195.15,671.36,203.8,1248.0,3980.0
1,19,4,2015,2015-04-19,01:00:00,3612.0,3612,0,0,0,...,139.18788,42.804626,113.00199,104.82534,final,195.15,671.36,203.8,1248.0,3980.0
2,19,4,2015,2015-04-19,02:00:00,3727.0,3727,0,0,0,...,139.18788,42.804626,113.00199,104.82534,final,195.15,671.36,203.8,1248.0,3980.0
3,19,4,2015,2015-04-19,03:00:00,3632.0,3632,0,0,0,...,139.18788,42.804626,113.00199,104.82534,final,195.15,671.36,203.8,1248.0,3980.0
4,19,4,2015,2015-04-19,04:00:00,3641.0,3641,0,0,0,...,139.18788,42.804626,113.00199,104.82534,final,195.15,671.36,203.8,1248.0,3980.0


In [68]:
final_data.to_csv("FINAL_V7.csv")